#### Statistical Hypothesis Testing (T-Test, ANOVA, Chi-Square)
---

##### Environment Setup and Data Loading

In [11]:
import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path
from typing import Any

# Load datasets and set significance threshold (alpha = 0.05)
CLEAN_DATA_DIR: Path = Path("../data/cleaned")

df_orders = pd.read_csv(CLEAN_DATA_DIR / "orders_features.csv")
df_items = pd.read_csv(CLEAN_DATA_DIR / "olist_order_items_cleaned.csv")
df_products = pd.read_csv(CLEAN_DATA_DIR / "olist_products_cleaned.csv")
df_payments = pd.read_csv(CLEAN_DATA_DIR / "olist_order_payments_cleaned.csv")
df_reviews = pd.read_csv(CLEAN_DATA_DIR / "olist_order_reviews_cleaned.csv")

alpha = 0.05

print(f"Loaded orders: {df_orders.shape}")
print(f"Loaded items: {df_items.shape}")
print(f"Loaded products: {df_products.shape}")
print(f"Loaded payments: {df_payments.shape}")
print(f"Loaded reviews: {df_reviews.shape}")
print(f"Significance threshold (alpha): {alpha}")


Loaded orders: (99441, 17)
Loaded items: (112650, 7)
Loaded products: (32951, 10)
Loaded payments: (103886, 5)
Loaded reviews: (99224, 7)
Significance threshold (alpha): 0.05


---
##### Test 1: One-Way ANOVA (Product Category vs Customer Spending)
Business Question: Do delayed orders receive statistically significantly lower review scores than on-time orders?

In [12]:
# 1. Merge delivery delay flag with customer review scores
df_orders_subset = df_orders[["order_id", "is_delayed"]]
df_reviews_subset = df_reviews[["order_id", "review_score"]]
df_order_reviews = pd.merge(df_orders_subset, df_reviews_subset, on="order_id", how="inner")

ontime_scores = df_order_reviews.loc[df_order_reviews["is_delayed"] == 0, "review_score"].dropna()
delayed_scores = df_order_reviews.loc[df_order_reviews["is_delayed"] == 1, "review_score"].dropna()

mean_ontime = float(ontime_scores.mean())
mean_delayed = float(delayed_scores.mean())

# 2. Two-sample Welch's t-test (unequal variances)
t_stat: Any
p_val: Any
t_stat, p_val = stats.ttest_ind(ontime_scores, delayed_scores, equal_var=False)

print(f"On-Time Deliveries Count : {len(ontime_scores):,} | Mean Score: {mean_ontime:.2f}")
print(f"Delayed Deliveries Count : {len(delayed_scores):,} | Mean Score: {mean_delayed:.2f}")
print(f"T-Statistic              : {float(t_stat):.4f}")
print(f"p-value                  : {float(p_val)}")

# 3. Hypothesis decision
if p_val < alpha:
    print(f"Decision: Reject Null Hypothesis (H0) because p-value ({float(p_val):.4e}) < {alpha}.")
    print("Conclusion: Delivery delay has a statistically significant negative impact on customer ratings.")
else:
    print(f"Decision: Fail to Reject Null Hypothesis (H0) because p-value >= {alpha}.")
    print("Conclusion: No statistically significant difference in ratings between on-time and delayed orders.")


On-Time Deliveries Count : 91,585 | Mean Score: 4.21
Delayed Deliveries Count : 7,639 | Mean Score: 2.55
T-Statistic              : 85.8114
p-value                  : 0.0
Decision: Reject Null Hypothesis (H0) because p-value (0.0000e+00) < 0.05.
Conclusion: Delivery delay has a statistically significant negative impact on customer ratings.


---
##### Test 2: One-Way ANOVA (Product Category vs Customer Spending)
Business Question: Does mean order spending differ significantly across top product categories?

In [13]:
# 1. Extract price series for top 5 product categories
df_items_subset = df_items[["product_id", "price"]]
df_products_subset = df_products[["product_id", "product_category_name_english"]]
df_item_cat = pd.merge(df_items_subset, df_products_subset, on="product_id", how="inner")

top_5_categories = df_item_cat["product_category_name_english"].value_counts().head(5).index.tolist()
category_price_groups = []

for category_name in top_5_categories:
    prices = df_item_cat.loc[df_item_cat["product_category_name_english"] == category_name, "price"].dropna()
    category_price_groups.append(prices)
    cat_mean = float(prices.mean())
    print(f"Category: {category_name:<25} | Items: {len(prices):>6,} | Mean Price: {cat_mean:>6.2f} BRL")

# 2. One-Way ANOVA across category price groups
f_stat: Any
p_val_anova: Any
f_stat, p_val_anova = stats.f_oneway(*category_price_groups)

print("-" * 65)
print(f"F-Statistic     : {float(f_stat):.4f}")
print(f"p-value         : {float(p_val_anova)}")

# 3. Hypothesis decision
if p_val_anova < alpha:
    print(f"Decision: Reject Null Hypothesis (H0) because p-value ({float(p_val_anova):.4e}) < {alpha}.")
    print("Conclusion: Mean customer spending differs significantly across product categories.")
else:
    print(f"Decision: Fail to Reject Null Hypothesis (H0) because p-value >= {alpha}.")
    print("Conclusion: No statistically significant difference in spending across product categories.")


Category: bed_bath_table            | Items: 11,115 | Mean Price:  93.30 BRL
Category: health_beauty             | Items:  9,670 | Mean Price: 130.16 BRL
Category: sports_leisure            | Items:  8,641 | Mean Price: 114.34 BRL
Category: furniture_decor           | Items:  8,334 | Mean Price:  87.56 BRL
Category: computers_accessories     | Items:  7,827 | Mean Price: 116.51 BRL
-----------------------------------------------------------------
F-Statistic     : 145.1869
p-value         : 1.4054652734311001e-123
Decision: Reject Null Hypothesis (H0) because p-value (1.4055e-123) < 0.05.
Conclusion: Mean customer spending differs significantly across product categories.


---
##### Test 3: Chi-Square Test of Independence (Payment Method vs Order Outcome)
Business Question: Is customer payment method statistically associated with order fulfillment

In [ ]:
# 1. Contingency table: Payment method vs order cancellation
df_orders_status = df_orders[["order_id", "order_status"]]
df_payments_type = df_payments[["order_id", "payment_type"]]
df_order_pay = pd.merge(df_orders_status, df_payments_type, on="order_id", how="inner")

df_filtered_pay = df_order_pay.loc[df_order_pay["order_status"].isin(["delivered", "canceled"])]
df_filtered_pay = df_filtered_pay.loc[df_filtered_pay["payment_type"] != "other"]

contingency_table = pd.crosstab(df_filtered_pay["payment_type"], df_filtered_pay["order_status"])

print("Observed Frequencies Contingency Table:")
print(contingency_table)

# 2. Chi-Square test of independence
chi2_stat: Any
p_val_chi2: Any
dof: Any
expected_frequencies: Any

chi2_stat, p_val_chi2, dof, expected_frequencies = stats.chi2_contingency(contingency_table)

print("-" * 65)
print(f"Chi-Square Statistic : {float(chi2_stat):.4f}")
print(f"Degrees of Freedom   : {int(dof)}")
print(f"p-value              : {float(p_val_chi2)}")

# 3. Hypothesis decision
if p_val_chi2 < alpha:
    print(f"Decision: Reject Null Hypothesis (H0) because p-value ({float(p_val_chi2):.4e}) < {alpha}.")
    print("Conclusion: Payment method and order outcome are statistically dependent.")
else:
    print(f"Decision: Fail to Reject Null Hypothesis (H0) because p-value >= {alpha}.")
    print("Conclusion: Payment method and order outcome are statistically independent.")

# 4. Cancellation rate breakdown by payment method
print("-" * 65)
print("Cancellation Rate per Payment Method:")

for payment_method in contingency_table.index:
    payment_row = contingency_table.loc[payment_method]
    delivered_method_orders = payment_row["delivered"]
    canceled_method_orders = payment_row["canceled"]
    total_method_orders = delivered_method_orders + canceled_method_orders
    cancel_rate = (canceled_method_orders / total_method_orders) * 100.0
    print(f"Payment Method: {payment_method:<15} | Orders: {total_method_orders:>6,} | Cancel Rate: {cancel_rate:>5.2f}%")


Observed Frequencies Contingency Table:
order_status  canceled  delivered
payment_type                     
boleto              95      19191
credit_card        444      74586
debit_card           7       1486
voucher            115       5493
-----------------------------------------------------------------
Chi-Square Statistic : 181.9691
Degrees of Freedom   : 3
p-value              : 3.3130277302290336e-39
Decision: Reject Null Hypothesis (H0) because p-value (3.3130e-39) < 0.05.
Conclusion: Payment method and order outcome are statistically dependent.
-----------------------------------------------------------------
Cancellation Rate per Payment Method:
Payment Method: boleto          | Orders: 19,286 | Cancel Rate:  0.49%
Payment Method: credit_card     | Orders: 75,030 | Cancel Rate:  0.59%
Payment Method: debit_card      | Orders:  1,493 | Cancel Rate:  0.47%
Payment Method: voucher         | Orders:  5,608 | Cancel Rate:  2.05%
